# Preprocessing and model feature preparation

This notebook demonstrates the two preprocessing stages used before model training. `prep_job.py` performs dataset-specific preparation and creates chronological train, validation, and test partitions. `sklearn_preprocessing.py` scales and encodes the selected model features.

The split is chronological: 70% train, 15% validation, and 15% test. The feature transformer is fitted on the training partition only. The complete dataset is large, so restart the kernel before rerunning this workflow if large DataFrames remain in memory.


In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model_data import RAW_MODEL_COLUMNS
from prep_job import (
    add_split_time,
    add_target,
    chronological_split,
    load_monthly_data,
    preprocess_partitions,
)
from sklearn_preprocessing import (
    make_feature_preprocessor,
    prepare_model_features,
)


## Load, label, and split the raw data

`add_target` creates the five-class target. `planned_time` uses planned arrival time and falls back to planned departure time. Rows without either timestamp are removed before chronological sorting.


In [2]:
data = load_monthly_data(PROJECT_ROOT / "data", RAW_MODEL_COLUMNS)
labelled_data = add_target(data)
split_ready_data = add_split_time(labelled_data)
partitions = chronological_split(split_ready_data)

split_summary = pd.DataFrame(
    {
        "partition": list(partitions),
        "rows": [len(partitions[name]) for name in partitions],
    }
)
split_summary["percent"] = (
    split_summary["rows"] / len(split_ready_data) * 100
).round(2)
split_summary


,partition,rows,percent
0,train,10231017,70.0
1,validation,2192361,15.0
2,test,2192361,15.0


## Apply dataset-specific preprocessing

Each partition is processed independently. The pipeline adds calendar features, removes extreme delay values, derives `first_station` from `train_line_ride_id`, removes rows without required route information, and drops technical source columns.


In [3]:
processed_partitions = preprocess_partitions(partitions)

preprocessing_summary = pd.DataFrame(
    {
        "partition": list(partitions),
        "rows_before": [len(partitions[name]) for name in partitions],
        "rows_after": [len(processed_partitions[name]) for name in processed_partitions],
    }
)
preprocessing_summary["removed"] = (
    preprocessing_summary["rows_before"]
    - preprocessing_summary["rows_after"]
)
preprocessing_summary


,partition,rows_before,rows_after,removed
0,train,10231017,9864275,366742
1,validation,2192361,2120356,72005
2,test,2192361,2126112,66249


In [4]:
for partition_name, partition in processed_partitions.items():
    print(f"{partition_name}: {partition.shape}")
    display(partition.head(2))


train: (9864275, 14)


,time,delay_in_min,arrival_is_canceled,train_type,line_number,station_name,train_number,train_line_station_num,final_destination_station,target,planned_time,day_of_week,planned_hour,first_station
0,2026-01-01 00:31:00,71.0,False,RE,RE30,Frankfurt (Main) Hbf,4168,1,Kassel Hbf,severe_or_cancelled,2025-12-31 23:20:00,Thursday,23,Frankfurt (Main) Hbf
1,2026-01-01 00:08:00,47.0,False,RE,RE70,Kiel Hbf,11236,8,NaN,severe_or_cancelled,2025-12-31 23:21:00,Thursday,23,Hamburg Hbf


validation: (2120356, 14)


,time,delay_in_min,arrival_is_canceled,train_type,line_number,station_name,train_number,train_line_station_num,final_destination_station,target,planned_time,day_of_week,planned_hour,first_station
10231017,2026-07-10 23:36:00,174.0,False,RE,RE42,Mönchengladbach Hbf,10158,21,NaN,severe_or_cancelled,2026-07-10 20:42:00,Friday,20,Münster (Westf) Hbf
10231018,2026-07-10 20:43:00,0.0,False,RE,RE32,Pegnitz,3092,4,NaN,on_time,2026-07-10 20:43:00,Friday,20,Bayreuth Hbf


test: (2126112, 14)


,time,delay_in_min,arrival_is_canceled,train_type,line_number,station_name,train_number,train_line_station_num,final_destination_station,target,planned_time,day_of_week,planned_hour,first_station
12423378,2026-08-21 17:22:00,12.0,False,ICE,NaN,Eisenach Hbf,1653,5,Dresden Hbf,delayed,2026-08-21 17:08:00,Friday,17,Frankfurt am Main Flughafen Fernbahnhof
12423380,2026-08-21 17:26:00,16.0,False,ICE,NaN,Erfurt Hbf,751,2,Berlin Gesundbrunnen,delayed,2026-08-21 17:08:00,Friday,17,Frankfurt (Main) Hbf


## Transform features for model training

`prepare_model_features` separates targets from model inputs. The sklearn transformer scales `train_line_station_num`, groups sparse and missing categories, and creates sparse one-hot features. Only the training partition is used in `fit_transform`; validation and test use `transform`.


In [5]:
train_features, y_train = prepare_model_features(processed_partitions["train"])
validation_features, y_validation = prepare_model_features(processed_partitions["validation"])
test_features, y_test = prepare_model_features(processed_partitions["test"])

preprocessor = make_feature_preprocessor()
X_train = preprocessor.fit_transform(train_features)
X_validation = preprocessor.transform(validation_features)
X_test = preprocessor.transform(test_features)

feature_summary = pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "rows": [X_train.shape[0], X_validation.shape[0], X_test.shape[0]],
        "encoded_features": [X_train.shape[1], X_validation.shape[1], X_test.shape[1]],
        "target_rows": [len(y_train), len(y_validation), len(y_test)],
    }
)
feature_summary


,partition,rows,encoded_features,target_rows
0,train,9864275,4758,9864275
1,validation,2120356,4758,2120356
2,test,2126112,4758,2126112


The resulting matrices can be passed to a model. The same workflow is available as one reusable function:

```python
from model_data import prepare_model_data

model_data = prepare_model_data(PROJECT_ROOT / "data")
model.fit(model_data.X_train, model_data.y_train)
```
